Neste exemplo estamos aplicando o método APU em um modelo LSTM. 
Para cada horizonte de previsão (2h, 4h, 8h, ..., 48h):

Carrega um modelo LSTM treinado.

Usa uma janela de entrada real (níveis do rio).

Calcula a previsão futura.

Propaga as incertezas dos dados de entrada e do modelo (APU).

Retorna: previsão ± incerteza expandida (com cobertura 95%).

Obs: entrar manualmente com uma sequancia qualquer de 10 valores 

In [ ]:
# apu_lstm_application.py
import tensorflow as tf
import pandas as pd
import numpy as np
import joblib
from datetime import datetime, timedelta
from apu_method import apu_uncertainty_lstm

# === Funções auxiliares === Calcula a matriz de correlação entre colunas de X, e retorna apenas a parte triangular superior, (M3)
def corr_upper_from_samples(X):
    R = np.corrcoef(X, rowvar=False)
    R = np.clip(R, -1.0, 1.0)
    np.fill_diagonal(R, 1.0)
    return np.triu(R, k=0)

def uncertainty_scale_factor(scaler): #Extrai o fator de escala aplicado pelo MinMaxScaler ou StandardScaler. Serve para converter incertezas em cm para a mesma escala do modelo.
    if hasattr(scaler, "data_max_") and hasattr(scaler, "data_min_"):
        return float(scaler.data_max_[0] - scaler.data_min_[0])
    if hasattr(scaler, "scale_"):
        return float(scaler.scale_[0])
    return 1.0

def cria_janelas(series, janela=10, atraso=48):
    #Função que monta pares (X, y) para treinamento/teste: 
    #X = valores passados com janela deslizante.
    # y = valor do alvo após certo atraso (horizonte de previsão).
    X, y = [], []
    for i in range(len(series) - janela - atraso):
        X.append(series[i:i+janela])
        y.append(series[i+janela+atraso-1])
    return np.array(X), np.array(y)

# === Configurações ===
# entrar com uma determinada sequencia de medições para prever o futuro : 
x_seq_real = np.array([[346], [347], [348], [349], [349], [350], [351], [352], [352], [353]], dtype=float) # sequencia de medições em um dado instante de tempo
u_x_seq_cm = np.array([5.0]*10) # estimativa da incerteza para cada medição 
u_y_cm = 5.0 
k_cov = 2.0
janela = 10
ultima_medicao_str = "2025-06-18 16:15:00"
ultima_medicao = datetime.strptime(ultima_medicao_str, "%Y-%m-%d %H:%M:%S")

# === Série histórica ===Combina data + hora, trata valores ausentes, interpola.
df1 = pd.read_excel("87382000-SAO LEOPOLDO.xlsx") # puxa o dataset
df1['datetime'] = pd.to_datetime(df1['Data'].astype(str) + ' ' + df1['Hora'].astype(str))
df1['Nível (cm)'] = pd.to_numeric(df1['Nível (cm)'], errors='coerce')
serie = df1[['datetime', 'Nível (cm)']].dropna().set_index('datetime')
serie['Nível (cm)'] = serie['Nível (cm)'].interpolate(method='linear').fillna(method='ffill')
serie_vals = serie[['Nível (cm)']].values

# === Horizontes de previsão ===
horizontes = {120: 8, 240: 16, 480: 32, 720: 48, 1440: 96, 2880: 192}
resultados = []

for minutos, atraso in horizontes.items():
    print(f"\nProcessando modelo {minutos} min...")

    # Carrega modelo e scaler correspondentes ao horizonte
    model = tf.keras.models.load_model(f"modelo_previsao_{minutos}min.h5", compile=False)
    scaler = joblib.load(f"scaler_previsao_{minutos}min.save")

    # Cria janelas históricas para cálculo de resíduos e correlações
    X_raw, y_raw = cria_janelas(serie_vals, janela=janela, atraso=atraso)
    X_flat = X_raw.reshape(len(X_raw), janela)
    y_raw = y_raw.reshape(-1, 1)

    X_scaled = scaler.transform(X_flat.reshape(-1, 1)).reshape(X_flat.shape)
    y_scaled = scaler.transform(y_raw)

    # Matriz de correlação triangular superior entre atrasos
    M3 = corr_upper_from_samples(X_scaled)

    # Resíduos em escala física
    X_model_in = X_scaled.reshape(len(X_scaled), janela, 1)
    y_pred_scaled = model.predict(X_model_in, verbose=0)
    y_pred_cm = scaler.inverse_transform(y_pred_scaled)
    y_cm = scaler.inverse_transform(y_scaled)
    residuos_cm = (y_pred_cm.flatten() - y_cm.flatten())
    SE_cm = float(np.mean(residuos_cm))
    RE_cm = float(np.std(residuos_cm - SE_cm))

    # Converte incertezas para escala do modelo
    s = uncertainty_scale_factor(scaler)
    u_x_seq_scaled = u_x_seq_cm / s
    u_y_scaled = u_y_cm / s
    SE_scaled = SE_cm / s
    RE_scaled = RE_cm / s

    # Entrada real escalada
    x_seq_scaled = scaler.transform(x_seq_real.reshape(-1, 1)).reshape(1, janela, 1)

    # Torna M3 simétrica
    R_corr = M3.copy()
    for i in range(R_corr.shape[0]):
        for j in range(i):
            R_corr[i, j] = R_corr[j, i]

    # --- APU ---
    out = apu_uncertainty_lstm(
        model=model,
        x_seq=x_seq_scaled.reshape(-1),
        u_x_seq=u_x_seq_scaled,
        SE=SE_scaled,
        RE=RE_scaled,
        u_y=u_y_scaled,
        k_coverage=k_cov,
        rho=R_corr
    )

    # Desserializa para cm
    y_pred_cm_single = scaler.inverse_transform([[out['y_pred']]])[0, 0]
    u_D_cm = out['u_D'] * s
    u_M_cm = out['u_M'] * s
    u_P_cm = out['u_P'] * s
    U_cm   = out['U'] * s

    hora_prevista = ultima_medicao + timedelta(minutes=minutos)

    resultados.append({
        'Data coleta': ultima_medicao,
        'Horizonte (min)': minutos,
        'Previsão para': hora_prevista,
        'Previsão (cm)': y_pred_cm_single,
        'u_D (dados) [cm]': u_D_cm,
        'u_M (modelo) [cm]': u_M_cm,
        'u_P (combinada) [cm]': u_P_cm,
        'U (expandida) [cm]': U_cm,
        'Previsão ± U': f"{y_pred_cm_single:.2f} ± {U_cm:.2f} cm"
    })

# === Resultado final ===
df_resultados = pd.DataFrame(resultados).sort_values("Horizonte (min)").reset_index(drop=True)
print("\nTabela final de previsões com incertezas:")
print(df_resultados)

# Opcional:
#df_resultados.to_excel("previsoes_apu_detalhadas.xlsx", index=False)
#df_resultados.to_csv("previsoes_apu_detalhadas.csv", index=False)


C:\Users\AdmPDI\AppData\Local\Temp\ipykernel_7552\909278202.py:48: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  serie['Nível (cm)'] = serie['Nível (cm)'].interpolate(method='linear').fillna(method='ffill')



Processando modelo 120 min...

Processando modelo 240 min...

Processando modelo 480 min...

Processando modelo 720 min...

Processando modelo 1440 min...

Processando modelo 2880 min...

Tabela final de previsões com incertezas:
          Data coleta  Horizonte (min)       Previsão para  Previsão (cm)  \
0 2025-06-18 16:15:00              120 2025-06-18 18:15:00     353.697861   
1 2025-06-18 16:15:00              240 2025-06-18 20:15:00     359.272309   
2 2025-06-18 16:15:00              480 2025-06-19 00:15:00     368.550743   
3 2025-06-18 16:15:00              720 2025-06-19 04:15:00     365.308173   
4 2025-06-18 16:15:00             1440 2025-06-19 16:15:00     394.628648   
5 2025-06-18 16:15:00             2880 2025-06-20 16:15:00     415.832054   

   u_D (dados) [cm]  u_M (modelo) [cm]  u_P (combinada) [cm]  \
0          7.029207           4.191207              8.183885   
1          7.096925           7.300641             10.181635   
2          7.221741          12.87935